# Compound hydroclimatic extremes in southern Vietnam

Exploratory walkthrough of the index chain: **ERA5 (or synthetic) -> SPEI / heat
exceedance -> compound masks -> hazard footprint -> trend**.

This notebook is the readable companion to `src/pipeline.py`. It runs against the
synthetic validation field by default so it executes without a CDS API key; flip
`USE_SYNTHETIC` to `False` once your ERA5 download is in `data/raw/`.

**A note on the synthetic field.** It is not a stand-in for real data and must never
be presented as a result. Its only purpose is to let the analysis code be exercised
against a signal whose answer is known: the generator imposes a warming rate and
reports it, so the trend estimator can be checked rather than believed.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

# Make src/ importable when the notebook is run from notebooks/.
sys.path.insert(0, str(Path.cwd().parent / "src"))

from fetch import SOUTHERN_VIETNAM, synthetic_field
from indices import (
    cell_area,
    compound_mask,
    event_catalogue,
    hargreaves_pet_monthly,
    hazard_footprint,
    heat_exceedance,
    mann_kendall_p,
    spei,
    theil_sen_trend,
)

USE_SYNTHETIC = True
print("xarray", xr.__version__)

## 1. Load the field

Study domain: Mekong Delta, Southeast region, and the southern Central Highlands
fringe — roughly 8.3–12.5°N, 104.0–107.8°E.

In [ ]:
if USE_SYNTHETIC:
    ds = synthetic_field(start="1991-01-01", end="2024-12-31")
else:
    from fetch import fetch_monthly
    ds = fetch_monthly("1991-01-01", "2024-12-31")

print(f"{ds.sizes['time']} months x {ds.sizes['lat']} lat x {ds.sizes['lon']} lon")
print("time span:", str(ds.time.values[0])[:10], "->", str(ds.time.values[-1])[:10])
print("imposed warming (synthetic only):", ds.attrs.get('warming_per_decade_K', 'n/a'))
ds

## 2. Build the hazard indices

Three pieces, each with a reason to exist:

- **SPEI-3** — precipitation *and* evaporative demand. A precipitation-only index
  (SPI) misses droughts driven by heat rather than rainfall deficit, which is
  precisely the compound case this work targets.
- **Tmax > reference-period p90** — a relative threshold, because 35 °C is extreme
  in the Central Highlands and unremarkable in the Delta.
- **Compound mask** — the *intersection* of the two. Union would count a dry month
  and a hot month in different seasons as one event.

In [ ]:
precip_mm = ds.tp * 1000.0
pet = hargreaves_pet_monthly(ds.t2m, xr.full_like(ds.t2m, 8.0), ds.lat)

spei3 = spei(precip_mm, pet, window=3)
hot = heat_exceedance(ds.t2m, 90)
hot = hot.assign_coords(time=ds.time)  # keep a single canonical time axis

drought = spei3 < spei3.sel(time=slice('1991-01-01','2020-12-31')).quantile(0.10, dim='time')
wet3 = precip_mm.rolling(time=3, min_periods=3).sum()
wet = wet3 > wet3.sel(time=slice('1991-01-01','2020-12-31')).quantile(0.90, dim='time')

compound_dh = compound_mask(drought, hot)
compound_wh = compound_mask(wet, hot)

summary = pd.Series({
    'SPEI-3 mean': float(spei3.mean()),
    'SPEI-3 std': float(spei3.std()),
    'drought cell-months': int(drought.sum()),
    'heat cell-months': int(hot.sum()),
    'compound drought-heat cell-months': int(compound_dh.sum()),
    'compound wet-heat cell-months': int(compound_wh.sum()),
})
summary

### Why the co-occurrence rate is the interesting number

Drought and heat are each stationary by construction — both thresholds are
percentiles of their own reference period, so each fires ~10 % of the time. If they
were independent, the compound event would fire ~1 % of the time and the ratio
below would sit near 1.0.

The ratio is the physical signal. A value materially above 1.0 means the drivers are
coupled — which is exactly the coupling that makes compound events worse than the
sum of their parts.

In [ ]:
n_cells = int(drought.isel(time=0).size)
n_months = int(drought.sizes['time'])
total = n_cells * n_months

p_drought = float(drought.sum()) / total
p_heat = float(hot.sum()) / total
p_comp = float(compound_dh.sum()) / total
p_indep = p_drought * p_heat

pd.Series({
    'P(drought)': round(p_drought, 4),
    'P(heat)': round(p_heat, 4),
    'P(compound) observed': round(p_comp, 4),
    'P(compound) if independent': round(p_indep, 4),
    'co-occurrence ratio': round(p_comp / p_indep, 3) if p_indep > 0 else np.nan,
})

## 3. Hazard footprint through time

The footprint is the area under simultaneous exceedance, in km². Two details that
change the answer and are easy to get wrong:

1. **Spherical cell area.** ERA5's 0.25° cells shrink with latitude. A constant-area
   assumption inflates every footprint statistic.
2. **Rolling mean for display only.** The monthly series has a strong seasonal cycle;
   the 12-month mean is drawn on top so the trend is visible. All statistics are
   computed on the unsmoothed series.

In [ ]:
area = cell_area(ds.lat, ds.lon)
print(f"domain area: {float(area.sum()):,.0f} km2")
print(f"cell area range: {float(area.min()):,.0f} - {float(area.max()):,.0f} km2")

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for ax, mask, colour, label in (
    (axes[0], compound_dh, '#c0392b', 'Drought-heat'),
    (axes[1], compound_wh, '#2471a3', 'Wet-heat'),
):
    fp = hazard_footprint(mask, area)
    ax.fill_between(fp.time.values, fp.values, color=colour, alpha=0.25, label='monthly')
    ax.plot(fp.time.values, fp.rolling(time=12, min_periods=6).mean().values,
            color=colour, lw=2, label='12-month mean')
    ax.set_ylabel('footprint (km²)')
    ax.set_title(f'Compound {label} hazard footprint')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Trends, with significance

Theil–Sen slope with a Mann–Kendall test, stippled where p < 0.05. At this
resolution there are enough cells that a non-trivial fraction will show a large
slope by chance — reporting a slope without its p-value is how spurious trends get
published.

In [ ]:
slope = theil_sen_trend(hot.astype(float))
pval = mann_kendall_p(hot.astype(float))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

im = axes[0].pcolormesh(slope.lon, slope.lat, slope, cmap='RdBu_r', shading='auto')
axes[0].contourf(pval.lon, pval.lat, (pval < 0.05).astype(float),
                 levels=[0.5, 1.5], colors='none', hatches=['...', ''])
axes[0].set_title('Trend in heat exceedance (per year)\nhatched = p < 0.05')
plt.colorbar(im, ax=axes[0], label='frequency per year')

fp_annual = hazard_footprint(compound_dh, area).resample(time='1YE').sum() / 1000.0
axes[1].bar(fp_annual.time.dt.year, fp_annual.values, color='#c0392b', alpha=0.8)
axes[1].set_title('Annual compound drought-heat footprint')
axes[1].set_ylabel('thousand km²·month')
axes[1].grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
frac_sig = float(np.nanmean((pval < 0.05).values))
print(f"mean slope: {float(np.nanmean(slope.values))*10:.4f} per decade")
print(f"fraction of cells significant at p<0.05: {frac_sig:.1%}")
print()
if USE_SYNTHETIC:
    imposed = ds.attrs.get('warming_per_decade_K')
    recovered = float(np.nanmean(theil_sen_trend(ds.t2m - 273.15).values)) * 10
    print(f"VALIDATION -- imposed {imposed} K/decade, recovered {recovered:.4f} K/decade")
    print(f"             error {abs(recovered-imposed):.4f} K/decade")

## 5. Event catalogue

Discrete events with start, end, duration and spatial extent.

**Known simplification.** Contiguity is assessed per grid cell in time, then
aggregated — not by connected-component labelling in space-time. A spatially
fragmented event spanning two regions will therefore be counted once. Fixing this
means a 4-D structure element and an explicit decision about whether diagonal
adjacency counts; `indices.label_spacetime()` is left as a documented stub so the
simplification is visible rather than buried.

In [ ]:
cat = event_catalogue(compound_dh, min_cells=5)
print(f"{len(cat)} compound drought-heat events (min 5 cells)")
cat.head(10)

## 6. What this repository does *not* do

Stated here so the boundary is explicit rather than inferred:

- **No socio-economic linkage.** No exposure, vulnerability, or observed-loss data.
  This is the hazard side of the chain only.
- **No daily-resolution indices.** At monthly resolution a 5-day flash drought does
  not register. Daily ETCCDI indices are the natural extension.
- **No downscaling or bias correction.** ERA5 is used as-is, at its native 0.25°.
- **No attribution.** Trends are described, not attributed to anthropogenic forcing.

The natural next step is the one this work is aimed at: joining these hazard
footprints to observed disaster losses and socio-economic vulnerability, and asking
which adaptation strategies remain effective when future conditions depart from
central projections.